#Install Libraries

In [1]:
!pip install transformers torch

#Import Libraries

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer



#Check Device

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


#Load Model

In [4]:
model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


#Response Function

In [5]:
def generate_response(user_input, chat_history_ids=None):

    user_input = "Answer clearly: " + user_input

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_k=40,
        top_p=0.85,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    response = response.split(".")[0] + "."

    return response, chat_history_ids

In [6]:
def smart_reply(user_input):
    text = user_input.lower()

    if "artificial intelligence" in text or "ai" in text:
        return "Artificial Intelligence is the simulation of human intelligence in machines."

    elif "machine learning" in text:
        return "Machine Learning is a subset of AI that allows systems to learn from data."

    elif "python" in text:
        return "Python was created by Guido van Rossum in 1991."

    return None

#Chat UI

In [7]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history = None

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # Check smart answers first
    fallback = smart_reply(user_input)

    if fallback:
        print("Chatbot:", fallback)
    else:
        response, chat_history = generate_response(user_input, chat_history)
        print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

User: hi
Chatbot: I'm confused.
User: what is artificial intelligence
Chatbot: Artificial Intelligence is the simulation of human intelligence in machines.
User: machine learning
Chatbot: Machine Learning is a subset of AI that allows systems to learn from data.
User: computer
Chatbot: I don't know what that is.
User: who founded python
Chatbot: Python was created by Guido van Rossum in 1991.
User: exit
Chatbot: Goodbye!
